In [1]:
from _src.Utils.CompositionLoader import CompositionExcelLoader
from _src.Composition.CompositionV2 import Composition
from _src.PhaseStability.TwoPhaseStabilityTestV3 import TwoPhaseStabilityTest
from _src.VLE.PhaseEquilibriumNewtonV2 import PhaseEquilibriumNewton
from _src.Utils.FluidPropertiesCalculatorV2 import FluidPropertiesCalculator
from _src.PhaseDiagram.SaturatuinPressureV2 import SaturationPressureCalculation
from _src.PhaseDiagram.PhaseEnvelopeV2 import PhaseDiagram

In [2]:
xl_loader = CompositionExcelLoader(r'/Users/daniilsidorenko/Desktop/PVT_TSU/diss/krsnln.xlsx')
comp_dict = xl_loader.load(header=True, sheet = 'to_model')

In [3]:
composition = Composition(comp_dict)
composition.evaluate_composition_data({'critical_temperature': 'kesler lee',
                                                    'critical_pressure': 'kesler lee',
                                                    'acentric_factor': 'riazi al sahhaf',
                                                    'critical_volume': 'hall yarborough',
                                                    'Kw': 'k watson',
                                                    'shift_parameter': 'jhaveri youngren'})

In [11]:
composition.edit_bip_for_components(component1='C1', component2='C2', bip = 0)
composition.edit_bip_for_components(component1='C1', component2='C3', bip = 0)
composition.edit_bip_for_components(component1='C1', component2='iC4', bip = 0)
composition.edit_bip_for_components(component1='C1', component2='nC4', bip = 0)
composition.edit_bip_for_components(component1='C1', component2='C2', bip = 0)

In [ ]:
composition.composition_data

In [4]:
condition_p = 1000
condition_t = 100 + 273.15

In [ ]:
phase_stab = TwoPhaseStabilityTest(composition, condition_p, condition_t)
phase_stab.calculate_phase_stability()
phase_stab.stable

In [ ]:
phase_stab.xi_l

In [ ]:
phase_stab.yi_v

In [ ]:
phase_equil = PhaseEquilibriumNewton(composition, condition_p, condition_t, phase_stab.k_values_for_flash)
flash_data = phase_equil.find_solve_loop()

In [ ]:
flash_data

Проблема: FluidProperties от Дениса будто некорректный, надо проверить на старом модуле

In [44]:
fluid_props = FluidPropertiesCalculator(composition=flash_data['xi_l'], composition_properties= composition.composition_data,
                                        p= condition_p, T=condition_t, eos_object = phase_equil.eos_liquid)

In [ ]:
fluid_props.density

In [ ]:
fluid_props.viscosity

In [ ]:
fluid_props.z_shift

### Работа через интерфейс  Model

In [4]:
from _src.CompositionalModel.CompositionalModelV2 import CompositionalModel
from _src.Utils.Conditions import Conditions

In [5]:
model = CompositionalModel(composition)

In [6]:
conds = Conditions(100, 330)

In [ ]:
model.flash(conditions=conds)

In [ ]:
model.composition.composition_data

In [ ]:
model.flashV2(conds)

In [ ]:
sat_p_new = SaturationPressureCalculation(composition_object = composition,
                                          p_max = 600, temp = 120)
sat_p_new.calculate_saturation_pressure()

In [12]:
from _src.PhaseDiagram.PhaseEnvelopeV2 import SaturationPressure

sat_p_new = SaturationPressure(composition, p_max=600, temp = 893)
sat_p_new.sp_convergence_loop()

phase stab Sv: 1.0077077834710868, Sl: 1.0001115172590151, stable: False
{'s_sp': 1.0077077834710868, 'y_sp': {'N2': 0.004882436358822996, 'CO2': 0.017504665516987202, 'C1': 0.5379514469466472, 'C2': 0.12573919517243237, 'C3': 0.08725122598804282, 'iC4': 0.010502634298470459, 'nC4': 0.03286667421007967, 'iC5': 0.008881698118678717, 'nC5': 0.017096303449784224, 'C6': 0.019019914617367965, 'C7': 0.025475084526476094, 'C8': 0.024260811059218356, 'C9': 0.018604375514549683, 'C10': 0.014800380807021545, 'C11': 0.011264037815401902, 'C12': 0.008578227832561257, 'C13': 0.00781826277054218, 'C14': 0.0063640753292248334, 'C15': 0.00481704157739105, 'C16': 0.003784767928545722, 'C17': 0.003010491938939188, 'C18': 0.002621364558793041, 'C19': 0.0022811793461084, 'C20': 0.0018834445138888399, 'C21': 0.001364758121750916, 'C22': 0.0012308763475792689, 'C23': 0.0009721095950946922, 'C24': 0.0008190457836640811, 'C25': 0.0007788111563539155, 'C26': 0.0005817164836256997, 'C27': 0.0005142546483574478,

328.125

In [ ]:
phase_env_new.plot_phase_diagram()

In [ ]:
phase_env_new.get_phase_diagram_data()

# Qwen Pcrit

In [ ]:
phase_env = PhaseEnvelopeTracer(composition)
phase_env.trace(T0=280, P0=5)

In [ ]:
import pandas as pd
history = phase_env.trace(T0=280.0, P0=5.0, h=2.0, max_steps=200)

df = pd.DataFrame(history)
print(f"Успешно пройдено точек: {len(df)}")
print(f"Последняя точка: T={df.iloc[-1]['T']:.2f} K, P={df.iloc[-1]['P']:.4f} bar")

In [ ]:
phase_env.branch

In [ ]:
tracer = PhaseEnvelopeTracer(composition, branch='bubble', debug=True)
history = tracer.trace(
    T0=280.0, 
    P0=5.0, 
    h=1.0,           # Меньший начальный шаг
    h_min=0.05,      # Более мелкий минимальный шаг
    max_steps=300,
    direction='increasing_T'  # Явно: растём по температуре
)

# Анализ результатов
df = pd.DataFrame(history)
print(f"\n✅ Успешно: {len(df)} точек")
print(f"Диапазон: T=[{df['T'].min():.1f}, {df['T'].max():.1f}] K, "
      f"P=[{df['P'].min():.2f}, {df['P'].max():.2f}] bar")

# График
if len(df) > 2:
    plt.figure(figsize=(9,6))
    plt.plot(df['T'], df['P'], 'o-', markersize=3, linewidth=1.5)
    plt.yscale('log')
    plt.xlabel('Temperature, K', fontsize=11)
    plt.ylabel('Pressure, bar (log)', fontsize=11)
    plt.grid(True, which='both', alpha=0.3)
    plt.title('Phase Envelope (Bubble Point)', fontsize=13)
    plt.tight_layout()
    plt.show()

In [107]:
import numpy as np
from _src.Composition.CompositionV2 import Composition
from _src.EOS.BrusilovskiyEOSV2 import BrusilovskiyEOS

class CriticalPointCalculator:
    """
    Стабилизированный расчет критической точки по Michelsen (1994).
    Устранены: осцилляции, зависимость от P_start, плохая обусловленность.
    """
    def __init__(self, composition: Composition, P_start: float = None):
        self.base_comp = composition
        self.components = tuple(composition.composition.keys())
        self.nc = len(self.components)
        self.z_arr = np.array([composition.composition[c] for c in self.components])
        
        # Данные компонент
        self._comp_data = composition.composition_data
        self._Tc_arr = np.array([self._comp_data['critical_temperature'][c] for c in self.components])
        self._Pc_arr = np.array([self._comp_data['critical_pressure'][c] for c in self.components])
        self._omega_arr = np.array([self._comp_data['acentric_factor'][c] for c in self.components])
        
        # Автоматический выбор P_start вблизи ожидаемой критической области
        if P_start is None:
            P_start = 0.6 * np.sum(self.z_arr * self._Pc_arr)  # ~30-50 атм для газов
        self.P_start = P_start
        self.trace_history = []

    def _create_eos(self, comp_dict: dict, T: float, P: float) -> BrusilovskiyEOS:
        eos = BrusilovskiyEOS(composition=self.base_comp.new_composition(comp_dict), p=P, t=T)
        eos.calc_eos()
        return eos

    def _get_ln_phi(self, eos: BrusilovskiyEOS, x_arr: np.ndarray, P: float) -> np.ndarray:
        return eos.fugacities - np.log(np.clip(x_arr, 1e-15, 1.0) * P)

    def _solve_bubble_point(self, P: float) -> tuple:
        T = np.clip(0.9 * np.sum(self.z_arr * self._Tc_arr), 150.0, 0.95 * np.max(self._Tc_arr))
        K = (self._Pc_arr / P) * np.exp(5.37 * (1.0 + self._omega_arr) * (1.0 - self._Tc_arr / T))
        K = np.clip(K, 1e-6, 1e6)

        for _ in range(50):
            y_arr = (self.z_arr * K) / np.sum(self.z_arr * K)
            eos_z = self._create_eos(dict(zip(self.components, self.z_arr)), T, P)
            eos_y = self._create_eos(dict(zip(self.components, y_arr)), T, P)
            
            K_new = np.exp(self._get_ln_phi(eos_z, self.z_arr, P) - self._get_ln_phi(eos_y, y_arr, P))
            K_new = np.clip(K_new, 1e-6, 1e6)
            if np.max(np.abs(K_new - K)) < 1e-4:
                K = K_new
                break
            K = K_new

            F = np.sum(self.z_arr * K) - 1.0
            if abs(F) < 1e-6: break

            dT = max(0.2, 1e-3 * T)
            K_p = np.exp(self._get_ln_phi(self._create_eos(dict(zip(self.components, self.z_arr)), T+dT, P), self.z_arr, P) - 
                         self._get_ln_phi(self._create_eos(dict(zip(self.components, y_arr)), T+dT, P), y_arr, P))
            dF_dT = (np.sum(self.z_arr * K_p) - 1.0 - F) / dT
            if abs(dF_dT) < 1e-8: T += np.sign(F) * 2.0
            else: T -= F / dF_dT
            T = np.clip(T, 150.0, 1.1 * np.max(self._Tc_arr))

        return T, y_arr, np.log(np.clip(K, 1e-12, 1e12))

    def _solve_theta(self, alpha: float, ln_K_ref: np.ndarray, beta: float = 0.5) -> float:
        theta = 0.0
        for _ in range(30):
            ln_K = alpha * ln_K_ref + theta
            K = np.exp(np.clip(ln_K, -50, 50))
            denom = 1.0 + beta * (K - 1.0)
            denom = np.where(np.abs(denom) < 1e-12, 1e-12 * np.sign(denom), denom)
            f = np.sum(self.z_arr * K / denom) - 1.0
            if abs(f) < 1e-10: return theta
            df_dtheta = np.sum(self.z_arr * K / denom - beta * self.z_arr * K**2 / denom**2)
            if abs(df_dtheta) < 1e-12: break
            theta -= f / df_dtheta
        return theta

    def _calc_f_and_jacobian(self, T: float, P: float, y_arr: np.ndarray) -> tuple:
        dict_y = dict(zip(self.components, y_arr))
        dict_z = dict(zip(self.components, self.z_arr))
        
        eos_y = self._create_eos(dict_y, T, P)
        eos_z = self._create_eos(dict_z, T, P)
        ln_phi_y = self._get_ln_phi(eos_y, y_arr, P)
        ln_phi_z = self._get_ln_phi(eos_z, self.z_arr, P)
        
        d = np.log(np.clip(y_arr, 1e-15, 1.0)) + ln_phi_y - np.log(np.clip(self.z_arr, 1e-15, 1.0)) - ln_phi_z
        f1 = np.dot(y_arr, d)
        f2 = np.dot(self.z_arr, d)
        
        # Логарифмическое масштабирование шагов: h = ε * T, ε * P
        hT, hP = 1e-4 * T, 1e-4 * P
        
        def get_dlnphi_dlnT(eos_obj, x_arr):
            lp_p = self._get_ln_phi(self._create_eos(dict(zip(self.components, x_arr)), T*(1+hT), P), x_arr, P)
            lp_m = self._get_ln_phi(self._create_eos(dict(zip(self.components, x_arr)), T*(1-hT), P), x_arr, P)
            return (lp_p - lp_m) / (2*hT)
            
        def get_dlnphi_dlnP(eos_obj, x_arr):
            lp_p = self._get_ln_phi(self._create_eos(dict(zip(self.components, x_arr)), T, P*(1+hP)), x_arr, P*(1+hP))
            lp_m = self._get_ln_phi(self._create_eos(dict(zip(self.components, x_arr)), T, P*(1-hP)), x_arr, P*(1-hP))
            return (lp_p - lp_m) / (2*hP)
            
        dlnphi_dlnT = get_dlnphi_dlnT(eos_y, y_arr) - get_dlnphi_dlnT(eos_z, self.z_arr)
        dlnphi_dlnP = get_dlnphi_dlnP(eos_y, y_arr) - get_dlnphi_dlnP(eos_z, self.z_arr)
        
        # Якобиан для переменных τ=ln T, π=ln P
        J11 = np.dot(y_arr, dlnphi_dlnT)
        J21 = np.dot(self.z_arr, dlnphi_dlnT)
        J12 = np.dot(y_arr, dlnphi_dlnP)
        J22 = np.dot(self.z_arr, dlnphi_dlnP)
        
        return f1, f2, J11, J12, J21, J22

    def calculate(self, alpha_steps=None, max_corrector_iters=15, tol=1e-7) -> dict:
        if alpha_steps is None:
            # Плотная фиксированная сетка. Адаптивная вставка удалена для стабильности итератора.
            alpha_steps = [1.0, 0.7, 0.5, 0.35, 0.25, 0.18, 0.12, 0.08, 0.05, 0.03]
            
        print("[CriticalPoint] Шаг 0: Поиск опорной точки (bubble-point)...")
        T, y_init, ln_K_ref = self._solve_bubble_point(self.P_start)
        P = self.P_start
        print(f"  → Начальная точка: T={T:.2f} K, P={P:.2f} atm")
        
        self.trace_history = [(1.0, T, P)]
        dT_dalpha, dP_dalpha = 0.0, 0.0  # Касательная для предиктора
        
        for i, alpha in enumerate(alpha_steps[1:], 1):
            theta = self._solve_theta(alpha, ln_K_ref, beta=0.5)
            ln_K = alpha * ln_K_ref + theta
            K = np.exp(np.clip(ln_K, -50, 50))
            y_arr = (self.z_arr * K) / np.sum(self.z_arr * K)
            
            # ✅ БЕЗОПАСНЫЙ Секантный предиктор
            if len(self.trace_history) >= 2:
                T_guess = T + dT_dalpha * (self.trace_history[-2][0] - alpha)
                P_guess = P + dP_dalpha * (self.trace_history[-2][0] - alpha)
            else:
                # На первом шаге используем текущую точку как приближение
                T_guess, P_guess = T, P
                
            T_guess = np.clip(T_guess, 150.0, 500.0)
            P_guess = np.clip(P_guess, 1.0, 300.0)

            # Корректор: Damped Newton в логарифмических переменных
            tau, pi = np.log(T_guess), np.log(P_guess)
            f_norm_prev = 1e9
            step_accept = 1.0
            
            for it in range(max_corrector_iters):
                f1, f2, J11, J12, J21, J22 = self._calc_f_and_jacobian(np.exp(tau), np.exp(pi), y_arr)
                f_norm = np.sqrt(f1**2 + f2**2)
                
                if it > 1 and f_norm < tol:
                    break
                    
                det = J11 * J22 - J12 * J21
                if abs(det) < 1e-14: det = 1e-14 * np.sign(det)
                
                dtau = -(J22 * f1 - J12 * f2) / det
                dpi = -(J11 * f2 - J21 * f1) / det
                
                # Trust-region: ограничиваем относительный шаг
                max_dtau, max_dpi = 0.05, 0.10  # ~5% по T, ~10% по P
                if abs(dtau) > max_dtau: dtau = max_dtau * np.sign(dtau)
                if abs(dpi) > max_dpi: dpi = max_dpi * np.sign(dpi)
                
                # Backtracking line search
                step = 1.0
                for _ in range(6):
                    tau_t = tau + step * dtau
                    pi_t = pi + step * dpi
                    T_t, P_t = np.exp(tau_t), np.exp(pi_t)
                    if T_t < 150.0 or P_t < 1.0:
                        step *= 0.5; continue
                    f1_t, f2_t, _, _, _, _ = self._calc_f_and_jacobian(T_t, P_t, y_arr)
                    if np.sqrt(f1_t**2 + f2_t**2) < f_norm:
                        break
                    step *= 0.5
                    
                tau += step * dtau
                pi += step * dpi
                
                if f_norm < f_norm_prev:
                    f_norm_prev = f_norm
                    step_accept = step
                else:
                    step = max(step * 0.2, 1e-4)
                    tau -= step * dtau * 0.5
                    pi -= step * dpi * 0.5

            T, P = np.exp(tau), np.exp(pi)
            
            # Обновляем касательную для следующего предиктора
            prev_alpha = self.trace_history[-1][0]
            dT_dalpha = (T - self.trace_history[-1][1]) / (alpha - prev_alpha)
            dP_dalpha = (P - self.trace_history[-1][2]) / (alpha - prev_alpha)
            
            self.trace_history.append((alpha, T, P))
            print(f"  → α={alpha:.3f}: T={T:.2f} K, P={P:.2f} atm |f|={f_norm:.2e} step_rel={step_accept:.2f}")

        # Квадратичная экстраполяция по α²
        points = self.trace_history[-3:]
        alpha_sq = np.array([p[0]**2 for p in points])
        T_vals = np.array([p[1] for p in points])
        P_vals = np.array([p[2] for p in points])
        
        A = np.vstack([np.ones(3), alpha_sq, alpha_sq**2]).T
        try:
            coeff_T = np.linalg.solve(A, T_vals)
            coeff_P = np.linalg.solve(A, P_vals)
            T_crit, P_crit = coeff_T[0], coeff_P[0]
        except np.linalg.LinAlgError:
            T_crit, P_crit = T_vals[-1], P_vals[-1]
            
        print(f"\n[CriticalPoint] КРИТИЧЕСКАЯ ТОЧКА:")
        print(f"  Tc = {T_crit:.3f} K, Pc = {P_crit:.3f} atm")
        return {'Tc': T_crit, 'Pc': P_crit, 'history': self.trace_history}

In [ ]:

# 2. Запуск расчета
calc = CriticalPointCalculator(composition, P_start=10.0)
result = calc.calculate()

print(f"Критическая точка: Tc = {result['Tc']:.2f} K, Pc = {result['Pc']:.2f} atm")

In [108]:
z = {'C1': 0.943, 'C2': 0.027, 'C3': 0.0074, 'nC4': 0.0049, 
     'nC5': 0.0027, 'C6': 0.0010, 'C7': 0.014}
compt = Composition(z, T_res=100)
compt.evaluate_composition_data({'critical_temperature': 'kesler lee',
                                                    'critical_pressure': 'pedersen',
                                                    'acentric_factor': 'riazi al sahhaf',
                                                    'critical_volume': 'hall yarborough',
                                                    'Kw': 'k watson',
                                                    'shift_parameter': 'jhaveri youngren'})

In [ ]:
calc = CriticalPointCalculator(compt, P_start=10)
result = calc.calculate()

print(f"Критическая точка: Tc = {result['Tc']:.2f} K, Pc = {result['Pc']:.2f} atm")